In [ ]:
import sys
import os
import time
import yaml
sys.path.append(os.path.abspath('..')) 
from drone_env.drone_sim import RoomDroneEnv
from stable_baselines3 import PPO
import pybullet as p

# ==========================================
# CONFIGURE WHICH STAGE TO WATCH LIVE:
STAGE_TO_WATCH = 0
# ==========================================

print(f"Live Tracker initialized. Monitoring Stage {STAGE_TO_WATCH}...")

# 1. Load stage parameters from the YAML config file
config_path = os.path.abspath(os.path.join('..', 'configs', 'teacher_ppo.yaml'))
with open(config_path, 'r') as file:
    config = yaml.safe_load(file)

stage_key = f"stage_{STAGE_TO_WATCH}"
stage_config = config['stages'][stage_key]
reward_weights = config['rewards']

NUM_OBS = stage_config['num_obstacles']
RAND_OBS = stage_config['randomize_obstacles']
RAND_COINS = stage_config['randomize_coins']
RUN_NAME = stage_config['run_name']

# 2. Initialize the environment parametrically
env = RoomDroneEnv(gui=True, 
                   num_obstacles=NUM_OBS, 
                   randomize_obstacles=RAND_OBS, 
                   randomize_coins=RAND_COINS, 
                   reward_weights=reward_weights)

# 3. Locate the best_model.zip inside the specific stage's folder
model_path = os.path.abspath(os.path.join('..', 'models', RUN_NAME, 'best_model'))
print(f"Waiting for continuous brain at: {model_path}.zip")

# Wait until the training script saves the very first model
while not os.path.exists(f"{model_path}.zip"):
    print("Waiting for the first EvalCallback to create best_model.zip...")
    time.sleep(5)

# 4. Load the initial model
model = PPO.load(model_path, env=env)
obs, info = env.reset()

# 5. Set a nice camera angle for monitoring
p.resetDebugVisualizerCamera(cameraDistance=6.0, cameraYaw=45, cameraPitch=-30, cameraTargetPosition=[0, 0, 1])

print(f"[{RUN_NAME}] Live Simulation started. Updating brain dynamically...")

# 6. Infinite evaluation loop with Live Updates
while True:
    # Agent decides based on its current memory
    action, _states = model.predict(obs, deterministic=True)
    obs, reward, terminated, truncated, info = env.step(action) 
    
    # Maintain real-time execution speed (240 Hz physics engine)
    time.sleep(1./240.) 
    
    # If the episode ends (crash, success, or timeout)
    if terminated or truncated:
        time.sleep(0.5) # Brief pause before resetting
        
        # LIVE UPDATE MAGIC: Try to fetch a newer model from the training script
        try:
            model = PPO.load(model_path, env=env)
        except Exception:
            # Silently pass if the file is currently being locked/written by train_teacher.py
            # The agent will just fly one more episode with the old brain.
            pass 
            
        obs, info = env.reset()